# Crossed Design and Response Similarity

::::::::{div}
:class: progress-analysis-app progress4-app

:::::::{div}
:class: progress2-section progress4-input

## Agenda

### Reframe the current regression as ANOVA

Use the [current regression in Progress 3](Progress-3.ipynb#regression-equation) as the starting point. Express the analysis in terms of prompt effects, temperature effects and their interaction, then define the corresponding ANOVA tests.

The current regression already treats prompt and temperature as categorical factors and includes their interaction. A factorial ANOVA can express the same linear model. The next step is to make the hypotheses and variance decomposition explicit.

### Compare responses across prompts using a crossed design

Let $P_1$ and $P_2$ be two prompt versions for the same question. Let $R_{ij}$ denote response $j$ generated from prompt $P_i$, and let $F(P_i,R_{kj})$ evaluate a response against prompt $P_i$. Cross the evaluation prompt with the source of the response: evaluate an answer generated from $P_2$ against $P_1$, then compare it with an answer generated from $P_1$.

::::::{admonition} Worked crossed design example
:class: tip ds-example-card

Generate repeated answers from each prompt:

$$
P_1 \longrightarrow R_{11},R_{12},R_{13},
\qquad
P_2 \longrightarrow R_{21},R_{22},R_{23}.
$$

Keep $P_1$ as the evaluation prompt and compare:

$$
F(P_1,R_{11})
\qquad\text{and}\qquad
F(P_1,R_{21}).
$$

The second score evaluates an answer generated from $P_2$ against $P_1$. How much does that score differ from the score for $R_{11}$?

Also compare the distance between two answers generated from $P_1$ with the distance between answers generated from different prompts:

$$
D(R_{11},R_{12})
\qquad\text{and}\qquad
D(R_{11},R_{21}).
$$

Does changing the prompt produce a larger difference than generating another answer from the same prompt? Extend the evaluation to $F(P_i,R_{kj})$ across evaluation prompts and response sources.
::::::

### Decide what to measure and connect it to ANOVA

Decide what we want to measure: the evaluation score of each answer, the difference between answers, or both. How would each measure connect to ANOVA? Use a few responses from $P_1$ and $P_2$ to make the comparison concrete.

### Find suitable modelling for scores bounded by 0 and 1

Research which statistical models are appropriate when scores lie in $[0,1]$ and the ideal score is 1. The aim is to model the bounded score and its relationship to prompt and response source, not to make the outcome follow a normal distribution.

Identify approaches that can represent scores near the upper limit and scores exactly equal to 1.

### Define text distance within the crossed design

Define whether distance should reflect wording, meaning or disagreement in factual content. Research semantic similarity and syntactic similarity as ways to compare responses. Look into sentiment analysis as a way to compare agreement and disagreement between answers.

Connect these methods to $D(R_{11},R_{12})$ and $D(R_{11},R_{21})$. Determine which measure can distinguish variation between repeated answers to the same prompt from changes associated with a different prompt. Relate these distances to the comparison between $F(P_1,R_{11})$ and $F(P_1,R_{21})$.

:::::::

:::::::{div}
:class: progress2-section progress4-anova

## The current model as ANOVA

The [Progress 3 regression](Progress-3.ipynb#regression-equation) already contains the required factors. For response $j$ from prompt $p$ at temperature $t$, let $\bar Y_{ptj}$ be its average over three evaluations:

$$
\bar Y_{ptj}=\mu+\alpha_p+\tau_t+(\alpha\tau)_{pt}+\varepsilon_{ptj}.
$$

Prompt has four levels and temperature has three. The prompt test compares means averaged equally over temperatures; the temperature test averages equally over prompts. The interaction test asks whether prompt differences change with temperature. Faithfulness and Answer Relevancy are fitted separately.

| Metric | Effect | Degrees of freedom | F | p |
| --- | --- | --- | --- | --- |
| Faithfulness | Prompt main effect | 3, 23 | 6.645 | 0.0021 |
| Faithfulness | Temperature main effect | 2, 23 | 1.168 | 0.3288 |
| Faithfulness | Prompt × Temperature interaction | 6, 23 | 1.057 | 0.4160 |
| Answer Relevancy | Prompt main effect | 3, 23 | 1.357 | 0.2806 |
| Answer Relevancy | Temperature main effect | 2, 23 | 3.418 | 0.0502 |
| Answer Relevancy | Prompt × Temperature interaction | 6, 23 | 0.294 | 0.9334 |

Each analysis contains 35 response averages. Original at temperature 0.6, response 2, lacks a complete evaluation triplet in the saved Progress 3 run. The unit is a generated response, not each evaluation.

#### Conclusion

* **Faithfulness:** the prompt test gives $p=0.0021$, but this result depends strongly on the three “Yes” answers scored zero. Removing them changes it to $p=0.0545$.
* **Answer Relevancy:** prompt differences are not clearly supported. The temperature result is borderline ($p=0.05017$); neither metric shows clear interaction evidence.

:::::{dropdown} Hypotheses, sums of squares and assumptions

The tests use Type III sums of squares with sum contrasts. With an interaction present, main effects refer to equal weighted marginal means. These are different hypotheses from the individual reference level coefficients in Progress 3.

$$
H_P:\alpha_1=\cdots=\alpha_4=0,\qquad
H_T:\tau_1=\tau_2=\tau_3=0,\qquad
H_{PT}:(\alpha\tau)_{pt}=0\ \text{for all }p,t.
$$

For each tested term, $F=(SS_{\mathrm{term}}/df_{\mathrm{term}})/(SSE/23)$. The total decomposition is:

| Metric | Total SS | Model SS | Residual SS |
| --- | --- | --- | --- |
| Faithfulness | 1.38627 | 0.77294 | 0.61333 |
| Answer Relevancy | 0.81601 | 0.29746 | 0.51855 |

The three adjusted Type III term sums need not add to the model SS because one cell has a missing response. Recoding the regression changes its coefficients, but its fitted values agree to numerical precision.

The classical F tests assume independent response errors with common variance and an approximately Gaussian error model. With only two or three responses per condition, these are exploratory working assumptions, not established properties of the data. The tests are not adjusted for multiple comparisons.

The three zero Faithfulness scores came from the judge treating “Yes” as a claim about oral argument recordings. That interpretation needs review. The exclusion above is a sensitivity check, not a replacement analysis. All results concern one underlying question.
:::::

:::::::

:::::::{div}
:class: progress2-section progress4-crossed

## A full crossed evaluation experiment

All four response sources are now crossed with all four evaluation prompts, separately at temperatures 0.3, 0.6 and 0.9. The 36 saved Qwen answers each receive three fresh evaluations under every prompt:

$$
\begin{aligned}
N_{\mathrm{answers}} &= 4\times3\times3 = 36,\\[6pt]
N_{\mathrm{evaluations}} &= 36\times4\times3 = 432.
\end{aligned}
$$

Let $R_{ktj}$ be answer $j$ generated from prompt $P_k$ at temperature $t$. Score $F(P_i,R_{ktj})$ evaluates that fixed answer against prompt $P_i$. Rows below change the response source; columns change the evaluation prompt. The diagonal evaluates answers with their own generation prompt. Every off diagonal cell is a crossed combination.

**417 of 432 evaluations passed validation.** There are 139 complete sets of three evaluations out of 144 planned answer and evaluation prompt combinations. A failed evaluation stays missing; it is never assigned a zero.

### Scores by response source and evaluation prompt

Each cell gives the mean of the available response averages. Parentheses show the number of complete answers out of three. Each response average requires all three evaluations to pass. Where counts differ, differences between cells may also reflect which answers are missing.

### Temperature 0.3

| Response source | Evaluate with Original | Evaluate with A | Evaluate with B | Evaluate with C |
| --- | --- | --- | --- | --- |
| Original | 0.767 (3/3) | 0.833 (3/3) | 0.767 (3/3) | 0.889 (3/3) |
| A | 0.963 (3/3) | 0.907 (3/3) | 0.915 (3/3) | 0.963 (3/3) |
| B | 0.841 (3/3) | 0.775 (3/3) | 0.775 (3/3) | 0.841 (3/3) |
| C | 0.944 (3/3) | 0.822 (3/3) | 0.811 (3/3) | 0.822 (3/3) |

### Temperature 0.6

| Response source | Evaluate with Original | Evaluate with A | Evaluate with B | Evaluate with C |
| --- | --- | --- | --- | --- |
| Original | 0.750 (2/3) | 0.750 (2/3) | 0.750 (2/3) | 0.750 (2/3) |
| A | 0.751 (3/3) | 0.818 (3/3) | 0.677 (3/3) | 0.818 (3/3) |
| B | 0.667 (3/3) | 0.667 (3/3) | 0.667 (3/3) | 0.667 (3/3) |
| C | 0.771 (3/3) | 0.771 (3/3) | 0.771 (3/3) | 0.771 (3/3) |

### Temperature 0.9

| Response source | Evaluate with Original | Evaluate with A | Evaluate with B | Evaluate with C |
| --- | --- | --- | --- | --- |
| Original | 0.679 (3/3) | 0.735 (3/3) | 0.735 (3/3) | 0.735 (3/3) |
| A | 0.740 (3/3) | 0.824 (2/3) | 0.754 (3/3) | 0.787 (3/3) |
| B | 0.708 (3/3) | 0.708 (3/3) | 0.708 (3/3) | 0.750 (3/3) |
| C | 0.843 (3/3) | 0.884 (3/3) | 0.843 (3/3) | 0.843 (3/3) |

### Conclusion

* 15 of 34 complete answers change score under at least one evaluation prompt.
* All 432 planned evaluations were attempted: 417 passed validation and 15 failed. Missing answers limit comparisons in the affected cells.
* The experiment still contains one underlying question. It tests these four prompt versions and saved answers, rather than establishing general performance across questions.

:::::{dropdown} All 36 response score rows

Each entry is an average of three valid evaluations. “Missing” means that at least one evaluation failed. Different prompt columns use the exact same response text.

| Temperature | Source | Response | Original | A | B | C |
| --- | --- | --- | --- | --- | --- | --- |
| 0.3 | A | 1 | 1.000 | 0.833 | 1.000 | 1.000 |
| 0.3 | A | 2 | 1.000 | 1.000 | 0.857 | 1.000 |
| 0.3 | A | 3 | 0.889 | 0.889 | 0.889 | 0.889 |
| 0.3 | B | 1 | 1.000 | 0.800 | 0.800 | 1.000 |
| 0.3 | B | 2 | 0.667 | 0.667 | 0.667 | 0.667 |
| 0.3 | B | 3 | 0.857 | 0.857 | 0.857 | 0.857 |
| 0.3 | C | 1 | 0.833 | 0.667 | 0.833 | 0.667 |
| 0.3 | C | 2 | 1.000 | 0.800 | 0.800 | 0.800 |
| 0.3 | C | 3 | 1.000 | 1.000 | 0.800 | 1.000 |
| 0.3 | Original | 1 | 0.800 | 1.000 | 0.800 | 1.000 |
| 0.3 | Original | 2 | 0.667 | 0.667 | 0.667 | 0.667 |
| 0.3 | Original | 3 | 0.833 | 0.833 | 0.833 | 1.000 |
| 0.6 | A | 1 | 0.600 | 0.800 | 0.600 | 0.800 |
| 0.6 | A | 2 | 0.875 | 0.875 | 0.875 | 0.875 |
| 0.6 | A | 3 | 0.778 | 0.778 | 0.556 | 0.778 |
| 0.6 | B | 1 | 0.500 | 0.500 | 0.500 | 0.500 |
| 0.6 | B | 2 | 1.000 | 1.000 | 1.000 | 1.000 |
| 0.6 | B | 3 | 0.500 | 0.500 | 0.500 | 0.500 |
| 0.6 | C | 1 | 0.800 | 0.800 | 0.800 | 0.800 |
| 0.6 | C | 2 | 0.714 | 0.714 | 0.714 | 0.714 |
| 0.6 | C | 3 | 0.800 | 0.800 | 0.800 | 0.800 |
| 0.6 | Original | 1 | 0.750 | 0.750 | 0.750 | 0.750 |
| 0.6 | Original | 2 | 0.750 | 0.750 | 0.750 | 0.750 |
| 0.6 | Original | 3 | Missing | Missing | Missing | Missing |
| 0.9 | A | 1 | 0.947 | 0.947 | 0.947 | 0.947 |
| 0.9 | A | 2 | 0.571 | Missing | 0.714 | 0.714 |
| 0.9 | A | 3 | 0.700 | 0.700 | 0.600 | 0.700 |
| 0.9 | B | 1 | 0.875 | 0.750 | 0.750 | 0.875 |
| 0.9 | B | 2 | 0.750 | 0.875 | 0.875 | 0.875 |
| 0.9 | B | 3 | 0.500 | 0.500 | 0.500 | 0.500 |
| 0.9 | C | 1 | 1.000 | 1.000 | 1.000 | 1.000 |
| 0.9 | C | 2 | 0.778 | 0.778 | 0.778 | 0.778 |
| 0.9 | C | 3 | 0.750 | 0.875 | 0.750 | 0.750 |
| 0.9 | Original | 1 | 0.571 | 0.571 | 0.571 | 0.571 |
| 0.9 | Original | 2 | 0.667 | 0.833 | 0.833 | 0.833 |
| 0.9 | Original | 3 | 0.800 | 0.800 | 0.800 | 0.800 |
:::::

:::::{dropdown} Failed evaluations

| Failure | Evaluations |
| --- | --- |
| AssertionError: 12 statements but 11 verdicts | 12 |
| AssertionError: 7 statements but 8 verdicts | 3 |

A statement count mismatch means that the judge returned a different number of verdicts from extracted statements. Such a score has an ambiguous denominator and is excluded. Raw attempts and error details are retained, with no retries chosen according to the resulting score.
:::::

:::::{dropdown} Prompts, metric and settings

**Original:** I need to do some legal research to be prepared for my oral argument. Can I visit the law library?

**A:** I'm preparing for an oral argument and need to conduct some legal research. Can I access the law library?

**B:** To be adequately prepared for my oral argument, I require some legal research. Is it possible to visit the law library?

**C:** I'm getting ready for an oral argument and need to do some research. Can I go to the law library?

The metric is DeepEval Answer Relevancy, which evaluates an answer against its input question. The current Faithfulness implementation uses the answer and retrieved context, so changing the evaluation prompt would not change its input.

All 432 attempts are fresh. The earlier Original and B pilot is retained separately and is not reused in these averages. Generation uses the saved Qwen3:4b answers. Evaluation uses local Llama 3.1 8B with temperature 0, context length 8192 and output limit 4096, through DeepEval 4.3.1. Each evaluation uses a fresh conversation. The fixed run order cycles evaluation prompts within answer, then answers within evaluation repeat.
:::::


:::::::

:::::::{div}
:class: progress2-section progress4-similarity

## Response similarity and agreement

### Agreement from a Python NLI model

The actual 36 saved answers are compared using [cross encoder NLI DeBERTa v3 small](https://huggingface.co/cross-encoder/nli-deberta-v3-small), run locally with Python and Transformers. These results replace the earlier assistant assigned labels.

Each answer pair is evaluated in both directions. **Entailment** means the premise supports the hypothesis, **contradiction** means they conflict, and **neutral** means the premise does not establish or contradict the hypothesis. A shorter answer may follow from a longer one even when the reverse does not hold.

The model returns three logits. Softmax converts them to scores that sum to 1, and the largest score supplies the predicted label. No manual claim categories or answer specific rules assign these labels.

| Pair relation | Within prompt | Between prompts | Total |
| --- | --- | --- | --- |
| Mutual entailment | 8 | 24 | 32 |
| One direction entails | 9 | 53 | 62 |
| Neutral both directions | 19 | 83 | 102 |
| Contradiction predicted | 0 | 2 | 2 |

“Mutual entailment” requires entailment in both directions. “Contradiction predicted” means at least one direction predicts contradiction. Otherwise a pair has entailment in one direction or neutral in both. These are model predictions of textual relations, not verified agreement labels or calibrated probabilities.

Both pairs with a contradiction prediction include a bare “Yes” answer. These predictions need the question context to be interpreted.

There are 198 pairs at matching temperatures and 396 directional predictions. All 36 answers are included. Reused answers mean the pairs are not independent observations.

### Actual examples of the four table rows

The numbers in the table count answer pairs. The first three examples use two generations from the same prompt; the fourth uses different prompts. Generation number identifies the saved repeated response within its prompt and temperature condition. Excerpts are shortened with […]; the model used the full answers shown in each disclosure. Scores beside the predictions are the model scores for the selected labels.

::::::{admonition} Worked example: Mutual entailment
:class: tip ds-example-card

| Answer | Generation prompt | Temperature | Generation number |
| --- | --- | --- | --- |
| Answer 1 | Original | 0.3 | 1 |
| Answer 2 | Original | 0.3 | 3 |

**Answer 1, excerpt:**

> Yes, you can visit the law library for legal research to prepare for your oral argument. […] You can find a law library in your area by clicking the link provided in the reference passage [2].

**Answer 2, excerpt:**

> Yes, you can visit a law library in California to do legal research for your oral argument. […] You can find a law library in your area by clicking the link provided in the reference passage [2].

**Model predictions:** Answer 1 → answer 2: **entailment** (0.979); answer 2 → answer 1: **entailment** (0.995).

The model predicts that each answer supports the other. Both describe library access, research resources and printing.

:::::{dropdown} Full answers used by the model

**Answer 1:**

> Yes, you can visit the law library for legal research to prepare for your oral argument. The law libraries in California offer self-help resources, including materials on reserve or at the reference desk, and often have printers to print legal information and forms. You can find a law library in your area by clicking the link provided in the reference passage [2].

**Answer 2:**

> Yes, you can visit a law library in California to do legal research for your oral argument. The California Courts have self-help law libraries that offer resources like legal materials on reserve, reference desk assistance, and printers for legal forms. You can find a law library in your area by clicking the link provided in the reference passage [2].
:::::

::::::

::::::{admonition} Worked example: One direction entails
:class: tip ds-example-card

| Answer | Generation prompt | Temperature | Generation number |
| --- | --- | --- | --- |
| Answer 1 | Original | 0.6 | 2 |
| Answer 2 | Original | 0.6 | 3 |

**Answer 1, excerpt:**

> Yes, you can visit a law library in California to do legal research for your oral argument. […] The passage [1] also mentions that each Court of Appeal has self-help resources online and you can contact your local Court of Appeal to see if they have a self-help center at the court.

**Answer 2, excerpt:**

> Yes, you can visit a law library in California to do legal research for your oral argument. […] The passage also states that law libraries 'can't give you legal advice, but they can suggest things that may be helpful.' Therefore, it is possible to visit a law library for legal research preparation for oral arguments.

**Model predictions:** Answer 1 → answer 2: **neutral** (0.988); answer 2 → answer 1: **entailment** (0.925).

The model predicts support from answer 2 to answer 1, but not in reverse. The full texts add different details: answer 1 mentions Court of Appeal resources; answer 2 says libraries cannot give legal advice. The directional prediction is shown as returned, not assumed to be correct.

:::::{dropdown} Full answers used by the model

**Answer 1:**

> Yes, you can visit a law library in California to do legal research for your oral argument. The reference passages indicate that California Courts have self-help resources including law libraries where you can access legal materials, forms, and information. The passage [2] describes several law libraries across California with contact information and websites. These libraries can suggest helpful materials and sometimes have printers for legal forms. The passage [1] also mentions that each Court of Appeal has self-help resources online and you can contact your local Court of Appeal to see if they have a self-help center at the court.

**Answer 2:**

> Yes, you can visit a law library in California to do legal research for your oral argument. The reference passages indicate that California Courts have self-help law libraries that provide legal information, materials on reserve, and reference services. For example, passage [2] lists multiple law libraries across California with contact information and websites, such as the Alameda County Bernard E. Witkin Law Library and others. These libraries can suggest helpful materials, including legal forms, and they often have printers to print legal information. The passage also states that law libraries 'can't give you legal advice, but they can suggest things that may be helpful.' Therefore, it is possible to visit a law library for legal research preparation for oral arguments.
:::::

::::::

::::::{admonition} Worked example: Neutral both directions
:class: tip ds-example-card

| Answer | Generation prompt | Temperature | Generation number |
| --- | --- | --- | --- |
| Answer 1 | Original | 0.3 | 1 |
| Answer 2 | Original | 0.3 | 2 |

**Answer 1, excerpt:**

> Yes, you can visit the law library for legal research to prepare for your oral argument. […] You can find a law library in your area by clicking the link provided in the reference passage [2].

**Answer 2, excerpt:**

> Yes, you can visit the law libraries in California to do legal research for your oral argument. […] You can find a law library in your area through the provided list of California County Law Libraries.

**Model predictions:** Answer 1 → answer 2: **neutral** (0.987); answer 2 → answer 1: **neutral** (0.993).

Both answers affirm access and mention similar resources. Nevertheless, the model predicts neutral in both directions. This example shows why neutral cannot be read as disagreement.

:::::{dropdown} Full answers used by the model

**Answer 1:**

> Yes, you can visit the law library for legal research to prepare for your oral argument. The law libraries in California offer self-help resources, including materials on reserve or at the reference desk, and often have printers to print legal information and forms. You can find a law library in your area by clicking the link provided in the reference passage [2].

**Answer 2:**

> Yes, you can visit the law libraries in California to do legal research for your oral argument. The law libraries offer self-help resources, including materials on reserve or at the reference desk, and often have printers to print legal information and forms. You can find a law library in your area through the provided list of California County Law Libraries.
:::::

::::::

::::::{admonition} Worked example: Contradiction predicted
:class: tip ds-example-card

| Answer | Generation prompt | Temperature | Generation number |
| --- | --- | --- | --- |
| Answer 1 | A | 0.9 | 3 |
| Answer 2 | B | 0.9 | 3 |

**Answer 1, excerpt:**

> Yes, you can access law libraries in California to conduct legal research for your oral argument. […] Law libraries can suggest helpful resources but cannot give legal advice.

**Answer 2, excerpt:**

> Yes

**Model predictions:** Answer 1 → answer 2: **contradiction** (0.917); answer 2 → answer 1: **contradiction** (0.934).

Answer 2 is only “Yes”. The model predicts contradiction when comparing the answer texts without their questions. This does not establish an actual disagreement about library access.

:::::{dropdown} Full answers used by the model

**Answer 1:**

> Yes, you can access law libraries in California to conduct legal research for your oral argument. The California Courts have self-help law libraries that provide legal information, materials on reserve, and access to legal forms. Each Court of Appeal has self-help resources online, and you can also contact your local Court of Appeal to see if they have a self-help center at the court. Law libraries can suggest helpful resources but cannot give legal advice.

**Answer 2:**

> Yes
:::::

::::::

### Sensitivity to question context

Three saved answers contain only “Yes”. Their meaning depends on the question, so the answer only comparison is repeated with the identical original question prepended to both answers. This adds context without manually rewriting any answer. Shared question text can itself affect the predictions.

| Pair relation | Answers only | Question included |
| --- | --- | --- |
| Mutual entailment | 32 | 46 |
| One direction entails | 62 | 40 |
| Neutral both directions | 102 | 111 |
| Contradiction predicted | 2 | 1 |

The pair category changes for 49 of 198 pairs. Both sets of predictions are retained; the question included result is a sensitivity check rather than an automatic correction.

### Distance calculation

Let $R_{11}$ and $R_{12}$ be two answers from the same prompt, and $R_{21}$ an answer from another prompt. Calculate $D(R_{11},R_{12})$ within a prompt and $D(R_{11},R_{21})$ between prompts, keeping generation temperature the same.

**Cosine similarity and cosine distance.** Let $u$ and $v$ be the vector representations of two complete answers. Cosine similarity compares their directions. Cosine distance is $1 - \text{cosine similarity}$:

$$
D_{\mathrm{cos}}(R_a,R_b)=1-\frac{u^\top v}{\lVert u\rVert\lVert v\rVert}.
$$

Identical vector directions give 0; larger values mean less similar embeddings. The theoretical range is 0 to 2. This is not a probability of disagreement. The question included version applies the same calculation to “Question: [original question] Answer: [answer]”.

**Syntactic distance proxy.** Tokenize each full answer and assign a part of speech tag to every token. Let $\mathrm{ED}$ count the minimum insertions, deletions and substitutions needed to turn one tag sequence into the other, then divide by the longer sequence length:

$$
D_{\mathrm{POS}}(R_a,R_b)=\frac{\mathrm{ED}(T_a,T_b)}{\max(|T_a|,|T_b|,1)}.
$$

Here $T_a$ and $T_b$ are the tag sequences. The range is 0 to 1. This compares grammatical patterns, not the truth or agreement of the answers.

::::::{admonition} Worked distance calculation
:class: tip ds-example-card

For Original response 1 and A response 1 at temperature 0.3, using the complete saved answers:

| Calculation | Result |
| --- | --- |
| Cosine similarity | $0.800640$ |
| Cosine distance | $1 - 0.800640 = 0.199360$ |
| POS sequence lengths | 68 and 69 tags |
| Minimum tag edits | 21 |
| POS distance | $21/69 = 0.304348$ |

These are the same two answers used in the crossed score comparison below.
::::::

Each temperature has 12 answers: four prompts with three answers each. There are 12 within prompt pairs and 54 between prompt pairs per temperature. Across three temperatures, the table below averages 36 within prompt distances and 162 between prompt distances.

| Distance | Within prompt | Between prompts |
| --- | --- | --- |
| Cosine distance | 0.178 | 0.276 |
| Cosine distance with the question included | 0.051 | 0.071 |
| POS sequence | 0.517 | 0.614 |

```{figure} figures/progress4/semantic_distance.svg
:alt: Mean cosine distance within and between prompts at each generation temperature.
Each point averages 12 within prompt pairs or 54 between prompt pairs at one temperature.
```

```{figure} figures/progress4/syntax_distance.svg
:alt: Mean syntactic proxy distance within and between prompts at each generation temperature.
Each point averages the POS sequence distances for answer pairs at one temperature.
```



### Connection to the full crossed results

The crossed experiment measures **Answer Relevancy**. For each answer pair, keep the evaluation prompt fixed and take the absolute difference between its two response averages. Average those differences over Original, A, B and C. Both answers must have all 12 evaluations valid, leaving 176 pairs from 34 answers.

| NLI pair relation | Complete pairs | Mean absolute Answer Relevancy gap |
| --- | --- | --- |
| Mutual entailment | 29 | 0.113 |
| One direction entails | 57 | 0.142 |
| Neutral both directions | 88 | 0.186 |
| Contradiction predicted | 2 | 0.226 |

```{figure} figures/progress4/nli-crossed.svg
:alt: Maximum NLI contradiction score versus mean absolute Answer Relevancy gap for 176 complete answer pairs.
Each point is a pair at the same generation temperature. The horizontal value is the larger contradiction score from the two NLI directions. The vertical value compares those answers under the same evaluation prompts. Some points overlap.
```

### Conclusion

* The model predicts mutual entailment for 32 pairs and contradiction in at least one direction for 2 pairs in the answer only analysis.
* The table and plot connect predicted textual relations with evaluation score differences. They do not treat an Answer Relevancy difference as proof of disagreement.

:::::{dropdown} Predictions for all 198 answer pairs

Answer IDs contain the generation prompt, temperature and response number. A to B uses the first answer as the premise and the second as the hypothesis; B to A reverses them.

| Answer A | Answer B | A to B | B to A | Pair relation |
| --- | --- | --- | --- | --- |
| original-0.3-1 | original-0.3-2 | neutral | neutral | Neutral both directions |
| original-0.3-1 | original-0.3-3 | entailment | entailment | Mutual entailment |
| original-0.3-1 | A-0.3-1 | entailment | entailment | Mutual entailment |
| original-0.3-1 | A-0.3-2 | entailment | entailment | Mutual entailment |
| original-0.3-1 | A-0.3-3 | neutral | neutral | Neutral both directions |
| original-0.3-1 | B-0.3-1 | entailment | entailment | Mutual entailment |
| original-0.3-1 | B-0.3-2 | entailment | neutral | One direction entails |
| original-0.3-1 | B-0.3-3 | neutral | neutral | Neutral both directions |
| original-0.3-1 | C-0.3-1 | neutral | entailment | One direction entails |
| original-0.3-1 | C-0.3-2 | entailment | entailment | Mutual entailment |
| original-0.3-1 | C-0.3-3 | entailment | entailment | Mutual entailment |
| original-0.3-2 | original-0.3-3 | entailment | entailment | Mutual entailment |
| original-0.3-2 | A-0.3-1 | entailment | neutral | One direction entails |
| original-0.3-2 | A-0.3-2 | entailment | entailment | Mutual entailment |
| original-0.3-2 | A-0.3-3 | neutral | neutral | Neutral both directions |
| original-0.3-2 | B-0.3-1 | entailment | neutral | One direction entails |
| original-0.3-2 | B-0.3-2 | entailment | neutral | One direction entails |
| original-0.3-2 | B-0.3-3 | neutral | neutral | Neutral both directions |
| original-0.3-2 | C-0.3-1 | neutral | entailment | One direction entails |
| original-0.3-2 | C-0.3-2 | entailment | entailment | Mutual entailment |
| original-0.3-2 | C-0.3-3 | neutral | neutral | Neutral both directions |
| original-0.3-3 | A-0.3-1 | entailment | neutral | One direction entails |
| original-0.3-3 | A-0.3-2 | entailment | entailment | Mutual entailment |
| original-0.3-3 | A-0.3-3 | neutral | neutral | Neutral both directions |
| original-0.3-3 | B-0.3-1 | entailment | neutral | One direction entails |
| original-0.3-3 | B-0.3-2 | neutral | neutral | Neutral both directions |
| original-0.3-3 | B-0.3-3 | neutral | neutral | Neutral both directions |
| original-0.3-3 | C-0.3-1 | neutral | neutral | Neutral both directions |
| original-0.3-3 | C-0.3-2 | entailment | entailment | Mutual entailment |
| original-0.3-3 | C-0.3-3 | entailment | entailment | Mutual entailment |
| original-0.6-1 | original-0.6-2 | entailment | entailment | Mutual entailment |
| original-0.6-1 | original-0.6-3 | neutral | neutral | Neutral both directions |
| original-0.6-1 | A-0.6-1 | entailment | neutral | One direction entails |
| original-0.6-1 | A-0.6-2 | neutral | neutral | Neutral both directions |
| original-0.6-1 | A-0.6-3 | neutral | entailment | One direction entails |
| original-0.6-1 | B-0.6-1 | neutral | neutral | Neutral both directions |
| original-0.6-1 | B-0.6-2 | neutral | neutral | Neutral both directions |
| original-0.6-1 | B-0.6-3 | neutral | neutral | Neutral both directions |
| original-0.6-1 | C-0.6-1 | neutral | entailment | One direction entails |
| original-0.6-1 | C-0.6-2 | entailment | entailment | Mutual entailment |
| original-0.6-1 | C-0.6-3 | neutral | entailment | One direction entails |
| original-0.6-2 | original-0.6-3 | neutral | entailment | One direction entails |
| original-0.6-2 | A-0.6-1 | entailment | neutral | One direction entails |
| original-0.6-2 | A-0.6-2 | neutral | neutral | Neutral both directions |
| original-0.6-2 | A-0.6-3 | neutral | entailment | One direction entails |
| original-0.6-2 | B-0.6-1 | neutral | neutral | Neutral both directions |
| original-0.6-2 | B-0.6-2 | entailment | entailment | Mutual entailment |
| original-0.6-2 | B-0.6-3 | neutral | neutral | Neutral both directions |
| original-0.6-2 | C-0.6-1 | entailment | entailment | Mutual entailment |
| original-0.6-2 | C-0.6-2 | neutral | entailment | One direction entails |
| original-0.6-2 | C-0.6-3 | entailment | entailment | Mutual entailment |
| original-0.6-3 | A-0.6-1 | entailment | neutral | One direction entails |
| original-0.6-3 | A-0.6-2 | entailment | entailment | Mutual entailment |
| original-0.6-3 | A-0.6-3 | neutral | neutral | Neutral both directions |
| original-0.6-3 | B-0.6-1 | neutral | neutral | Neutral both directions |
| original-0.6-3 | B-0.6-2 | neutral | neutral | Neutral both directions |
| original-0.6-3 | B-0.6-3 | neutral | neutral | Neutral both directions |
| original-0.6-3 | C-0.6-1 | neutral | neutral | Neutral both directions |
| original-0.6-3 | C-0.6-2 | neutral | neutral | Neutral both directions |
| original-0.6-3 | C-0.6-3 | neutral | neutral | Neutral both directions |
| original-0.9-1 | original-0.9-2 | neutral | neutral | Neutral both directions |
| original-0.9-1 | original-0.9-3 | neutral | neutral | Neutral both directions |
| original-0.9-1 | A-0.9-1 | entailment | neutral | One direction entails |
| original-0.9-1 | A-0.9-2 | entailment | neutral | One direction entails |
| original-0.9-1 | A-0.9-3 | entailment | entailment | Mutual entailment |
| original-0.9-1 | B-0.9-1 | entailment | neutral | One direction entails |
| original-0.9-1 | B-0.9-2 | entailment | neutral | One direction entails |
| original-0.9-1 | B-0.9-3 | neutral | neutral | Neutral both directions |
| original-0.9-1 | C-0.9-1 | neutral | neutral | Neutral both directions |
| original-0.9-1 | C-0.9-2 | entailment | neutral | One direction entails |
| original-0.9-1 | C-0.9-3 | entailment | neutral | One direction entails |
| original-0.9-2 | original-0.9-3 | neutral | entailment | One direction entails |
| original-0.9-2 | A-0.9-1 | neutral | neutral | Neutral both directions |
| original-0.9-2 | A-0.9-2 | neutral | neutral | Neutral both directions |
| original-0.9-2 | A-0.9-3 | neutral | neutral | Neutral both directions |
| original-0.9-2 | B-0.9-1 | entailment | neutral | One direction entails |
| original-0.9-2 | B-0.9-2 | neutral | neutral | Neutral both directions |
| original-0.9-2 | B-0.9-3 | neutral | neutral | Neutral both directions |
| original-0.9-2 | C-0.9-1 | neutral | neutral | Neutral both directions |
| original-0.9-2 | C-0.9-2 | neutral | neutral | Neutral both directions |
| original-0.9-2 | C-0.9-3 | entailment | entailment | Mutual entailment |
| original-0.9-3 | A-0.9-1 | neutral | neutral | Neutral both directions |
| original-0.9-3 | A-0.9-2 | entailment | neutral | One direction entails |
| original-0.9-3 | A-0.9-3 | neutral | neutral | Neutral both directions |
| original-0.9-3 | B-0.9-1 | entailment | neutral | One direction entails |
| original-0.9-3 | B-0.9-2 | entailment | neutral | One direction entails |
| original-0.9-3 | B-0.9-3 | neutral | neutral | Neutral both directions |
| original-0.9-3 | C-0.9-1 | neutral | entailment | One direction entails |
| original-0.9-3 | C-0.9-2 | neutral | neutral | Neutral both directions |
| original-0.9-3 | C-0.9-3 | entailment | neutral | One direction entails |
| A-0.3-1 | A-0.3-2 | neutral | entailment | One direction entails |
| A-0.3-1 | A-0.3-3 | neutral | neutral | Neutral both directions |
| A-0.3-1 | B-0.3-1 | entailment | entailment | Mutual entailment |
| A-0.3-1 | B-0.3-2 | entailment | entailment | Mutual entailment |
| A-0.3-1 | B-0.3-3 | neutral | neutral | Neutral both directions |
| A-0.3-1 | C-0.3-1 | neutral | entailment | One direction entails |
| A-0.3-1 | C-0.3-2 | neutral | entailment | One direction entails |
| A-0.3-1 | C-0.3-3 | neutral | entailment | One direction entails |
| A-0.3-2 | A-0.3-3 | neutral | neutral | Neutral both directions |
| A-0.3-2 | B-0.3-1 | entailment | neutral | One direction entails |
| A-0.3-2 | B-0.3-2 | entailment | neutral | One direction entails |
| A-0.3-2 | B-0.3-3 | neutral | neutral | Neutral both directions |
| A-0.3-2 | C-0.3-1 | neutral | neutral | Neutral both directions |
| A-0.3-2 | C-0.3-2 | entailment | entailment | Mutual entailment |
| A-0.3-2 | C-0.3-3 | entailment | entailment | Mutual entailment |
| A-0.3-3 | B-0.3-1 | neutral | neutral | Neutral both directions |
| A-0.3-3 | B-0.3-2 | neutral | neutral | Neutral both directions |
| A-0.3-3 | B-0.3-3 | neutral | neutral | Neutral both directions |
| A-0.3-3 | C-0.3-1 | neutral | neutral | Neutral both directions |
| A-0.3-3 | C-0.3-2 | neutral | neutral | Neutral both directions |
| A-0.3-3 | C-0.3-3 | neutral | neutral | Neutral both directions |
| A-0.6-1 | A-0.6-2 | neutral | neutral | Neutral both directions |
| A-0.6-1 | A-0.6-3 | neutral | entailment | One direction entails |
| A-0.6-1 | B-0.6-1 | neutral | neutral | Neutral both directions |
| A-0.6-1 | B-0.6-2 | neutral | entailment | One direction entails |
| A-0.6-1 | B-0.6-3 | neutral | neutral | Neutral both directions |
| A-0.6-1 | C-0.6-1 | neutral | neutral | Neutral both directions |
| A-0.6-1 | C-0.6-2 | neutral | entailment | One direction entails |
| A-0.6-1 | C-0.6-3 | neutral | entailment | One direction entails |
| A-0.6-2 | A-0.6-3 | neutral | neutral | Neutral both directions |
| A-0.6-2 | B-0.6-1 | neutral | neutral | Neutral both directions |
| A-0.6-2 | B-0.6-2 | neutral | neutral | Neutral both directions |
| A-0.6-2 | B-0.6-3 | neutral | neutral | Neutral both directions |
| A-0.6-2 | C-0.6-1 | neutral | neutral | Neutral both directions |
| A-0.6-2 | C-0.6-2 | neutral | neutral | Neutral both directions |
| A-0.6-2 | C-0.6-3 | neutral | neutral | Neutral both directions |
| A-0.6-3 | B-0.6-1 | neutral | neutral | Neutral both directions |
| A-0.6-3 | B-0.6-2 | neutral | neutral | Neutral both directions |
| A-0.6-3 | B-0.6-3 | neutral | neutral | Neutral both directions |
| A-0.6-3 | C-0.6-1 | entailment | neutral | One direction entails |
| A-0.6-3 | C-0.6-2 | entailment | neutral | One direction entails |
| A-0.6-3 | C-0.6-3 | entailment | neutral | One direction entails |
| A-0.9-1 | A-0.9-2 | neutral | neutral | Neutral both directions |
| A-0.9-1 | A-0.9-3 | neutral | entailment | One direction entails |
| A-0.9-1 | B-0.9-1 | neutral | neutral | Neutral both directions |
| A-0.9-1 | B-0.9-2 | entailment | neutral | One direction entails |
| A-0.9-1 | B-0.9-3 | neutral | neutral | Neutral both directions |
| A-0.9-1 | C-0.9-1 | neutral | neutral | Neutral both directions |
| A-0.9-1 | C-0.9-2 | entailment | neutral | One direction entails |
| A-0.9-1 | C-0.9-3 | entailment | neutral | One direction entails |
| A-0.9-2 | A-0.9-3 | neutral | neutral | Neutral both directions |
| A-0.9-2 | B-0.9-1 | entailment | entailment | Mutual entailment |
| A-0.9-2 | B-0.9-2 | entailment | entailment | Mutual entailment |
| A-0.9-2 | B-0.9-3 | neutral | neutral | Neutral both directions |
| A-0.9-2 | C-0.9-1 | neutral | neutral | Neutral both directions |
| A-0.9-2 | C-0.9-2 | neutral | neutral | Neutral both directions |
| A-0.9-2 | C-0.9-3 | entailment | neutral | One direction entails |
| A-0.9-3 | B-0.9-1 | entailment | neutral | One direction entails |
| A-0.9-3 | B-0.9-2 | neutral | neutral | Neutral both directions |
| A-0.9-3 | B-0.9-3 | contradiction | contradiction | Contradiction predicted |
| A-0.9-3 | C-0.9-1 | neutral | neutral | Neutral both directions |
| A-0.9-3 | C-0.9-2 | entailment | neutral | One direction entails |
| A-0.9-3 | C-0.9-3 | neutral | neutral | Neutral both directions |
| B-0.3-1 | B-0.3-2 | neutral | entailment | One direction entails |
| B-0.3-1 | B-0.3-3 | neutral | neutral | Neutral both directions |
| B-0.3-1 | C-0.3-1 | neutral | entailment | One direction entails |
| B-0.3-1 | C-0.3-2 | neutral | entailment | One direction entails |
| B-0.3-1 | C-0.3-3 | neutral | entailment | One direction entails |
| B-0.3-2 | B-0.3-3 | neutral | neutral | Neutral both directions |
| B-0.3-2 | C-0.3-1 | neutral | neutral | Neutral both directions |
| B-0.3-2 | C-0.3-2 | neutral | entailment | One direction entails |
| B-0.3-2 | C-0.3-3 | neutral | entailment | One direction entails |
| B-0.3-3 | C-0.3-1 | neutral | neutral | Neutral both directions |
| B-0.3-3 | C-0.3-2 | neutral | neutral | Neutral both directions |
| B-0.3-3 | C-0.3-3 | neutral | neutral | Neutral both directions |
| B-0.6-1 | B-0.6-2 | neutral | neutral | Neutral both directions |
| B-0.6-1 | B-0.6-3 | entailment | entailment | Mutual entailment |
| B-0.6-1 | C-0.6-1 | neutral | neutral | Neutral both directions |
| B-0.6-1 | C-0.6-2 | neutral | neutral | Neutral both directions |
| B-0.6-1 | C-0.6-3 | neutral | neutral | Neutral both directions |
| B-0.6-2 | B-0.6-3 | neutral | neutral | Neutral both directions |
| B-0.6-2 | C-0.6-1 | entailment | neutral | One direction entails |
| B-0.6-2 | C-0.6-2 | neutral | neutral | Neutral both directions |
| B-0.6-2 | C-0.6-3 | entailment | entailment | Mutual entailment |
| B-0.6-3 | C-0.6-1 | neutral | neutral | Neutral both directions |
| B-0.6-3 | C-0.6-2 | neutral | neutral | Neutral both directions |
| B-0.6-3 | C-0.6-3 | neutral | neutral | Neutral both directions |
| B-0.9-1 | B-0.9-2 | entailment | entailment | Mutual entailment |
| B-0.9-1 | B-0.9-3 | entailment | entailment | Mutual entailment |
| B-0.9-1 | C-0.9-1 | neutral | entailment | One direction entails |
| B-0.9-1 | C-0.9-2 | neutral | neutral | Neutral both directions |
| B-0.9-1 | C-0.9-3 | neutral | entailment | One direction entails |
| B-0.9-2 | B-0.9-3 | neutral | neutral | Neutral both directions |
| B-0.9-2 | C-0.9-1 | neutral | neutral | Neutral both directions |
| B-0.9-2 | C-0.9-2 | neutral | neutral | Neutral both directions |
| B-0.9-2 | C-0.9-3 | neutral | neutral | Neutral both directions |
| B-0.9-3 | C-0.9-1 | neutral | neutral | Neutral both directions |
| B-0.9-3 | C-0.9-2 | contradiction | contradiction | Contradiction predicted |
| B-0.9-3 | C-0.9-3 | neutral | neutral | Neutral both directions |
| C-0.3-1 | C-0.3-2 | entailment | neutral | One direction entails |
| C-0.3-1 | C-0.3-3 | entailment | neutral | One direction entails |
| C-0.3-2 | C-0.3-3 | entailment | entailment | Mutual entailment |
| C-0.6-1 | C-0.6-2 | neutral | neutral | Neutral both directions |
| C-0.6-1 | C-0.6-3 | entailment | entailment | Mutual entailment |
| C-0.6-2 | C-0.6-3 | neutral | neutral | Neutral both directions |
| C-0.9-1 | C-0.9-2 | neutral | neutral | Neutral both directions |
| C-0.9-1 | C-0.9-3 | entailment | neutral | One direction entails |
| C-0.9-2 | C-0.9-3 | neutral | neutral | Neutral both directions |
:::::

:::::{dropdown} Answer pairs with the highest contradiction scores

The following are model predictions shown alongside the actual texts for inspection.

**B-0.9-3 and C-0.9-2**

**Answer A:** Yes

**Answer B:** Yes, you can go to a law library in California for research. According to reference passage [2], law libraries can suggest helpful materials, have printers for legal forms, and provide access to legal information. The passage lists several law libraries across California, including Alameda County, Orange County, and others, with contact information and websites. The law libraries do not provide legal advice but can assist with research and resources.

A to B: contradiction; B to A: contradiction. Maximum contradiction score: 0.935.

**A-0.9-3 and B-0.9-3**

**Answer A:** Yes, you can access law libraries in California to conduct legal research for your oral argument. The California Courts have self-help law libraries that provide legal information, materials on reserve, and access to legal forms. Each Court of Appeal has self-help resources online, and you can also contact your local Court of Appeal to see if they have a self-help center at the court. Law libraries can suggest helpful resources but cannot give legal advice.

**Answer B:** Yes

A to B: contradiction; B to A: contradiction. Maximum contradiction score: 0.934.

**A-0.9-3 and B-0.9-2**

**Answer A:** Yes, you can access law libraries in California to conduct legal research for your oral argument. The California Courts have self-help law libraries that provide legal information, materials on reserve, and access to legal forms. Each Court of Appeal has self-help resources online, and you can also contact your local Court of Appeal to see if they have a self-help center at the court. Law libraries can suggest helpful resources but cannot give legal advice.

**Answer B:** Yes, it is possible to visit the law libraries in California. Each Court of Appeal has self-help resources online and you can also contact your local Court of Appeal to see if they have a self-help center at the court. There are multiple law libraries listed in California with contact information and addresses, such as Alameda County Bernard E. Witkin Law Library, Orange County Public Law Library, and others.

A to B: neutral; B to A: neutral. Maximum contradiction score: 0.161.
:::::

:::::{dropdown} Python method and reproducibility

Model revision: `fa2804872c3b4bd748f38c0185cc85775361e735`. The model was trained on SNLI and MultiNLI. Inference uses CPU, evaluation mode and no text truncation. Both modes together produce 792 directional predictions. Full input texts, logits, softmax scores, labels, model revision and source hashes are saved with the analysis.

```python
with torch.inference_mode():
    logits = model(**tokens).logits
scores = logits.softmax(dim=-1)
labels = [model.config.id2label[i] for i in scores.argmax(dim=-1).tolist()]
```

Cosine distance is calculated as one minus cosine similarity between normalized answer vectors. The syntax proxy uses [NLTK POS tags](https://www.nltk.org/api/nltk.tag.perceptron.html). Neither embedding similarity nor sentiment polarity supplies the NLI labels. The model has not been validated against human labels for this question. A simple check also exposed an error: given “A person is eating pizza”, it predicted contradiction for “The person owns a bicycle”, although that information is simply unstated. This is a reason to inspect the predictions rather than treat them as ground truth.
:::::


:::::::

::::::::
